# Data Cleaning - Capa SILVER

En este notebook se realiza la transformación y limpieza de los datos para la capa SILVER.

El objetivo es preparar la información proveniente de la capa BRONZE para su posterior análisis y explotación.

In [1]:
import os
import sys

# Configurar variables de entorno para Hadoop en Windows
HADOOP_HOME = os.environ.get("HADOOP_HOME", "C:\\hadoop")

os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["hadoop.home.dir"] = HADOOP_HOME
os.environ["PATH"] = f"{os.environ.get('PATH', '')};{HADOOP_HOME}\\bin"

#Configurar Python para Spark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
# Importar librerías

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, trim, lower, regexp_replace, to_timestamp, coalesce, try_to_timestamp, lit

## Crear la sesión de Apache Spark

Se crea una sesión de Spark para el procesamiento distribuido de datos.

In [3]:
# Crear sesión de Spark

spark = (
    SparkSession.builder
    .appName("FinancialDigitalTwin_Silver")
    .getOrCreate()
)

print("Spark inicializado")

Spark inicializado


## Cargar los conjuntos de datos

In [4]:
# Cargar datasets

INPUT_PATH = "../data/processed/bronze"
OUTPUT_PATH = "../data/processed/silver"

# Crear directorio SILVER si no existe

import os

os.makedirs(OUTPUT_PATH, exist_ok=True)

users_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/users.parquet")
)

cards_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/cards.parquet")
)

transactions_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/transactions.parquet")
)

print("Datos cargados")

Datos cargados


## Explorar la información

In [5]:
# Mostrar primeras filas

users_spark.show(5)
cards_spark.show(5)
transactions_spark.show(5)

+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
| 825|         53|            66|      1966|         11|female|       462 rose lane|   34.15|  -117.76|          29278.0|      59696.0|  127613.0|         787|               5|
|1746|         53|            68|      1966|         12|female|3606 federal boul...|   40.76|   -73.74|          37891.0|      77254.0|  191349.0|         701|               5|
|1718|         81|            67|      1938|         11|female|     766 third drive|   34.02|  -117.89|          22

In [6]:
# Mostrar esquema

users_spark.printSchema()
cards_spark.printSchema()
transactions_spark.printSchema()

root
 |-- id: integer (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- per_capita_income: float (nullable = true)
 |-- yearly_income: float (nullable = true)
 |-- total_debt: float (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- client_id: integer (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: double (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: integer (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- credit_limit: doubl

## Transformar los datos

In [7]:
# Transformar users

users_silver = users_spark.withColumn(
    "gender",
    trim(lower(col("gender")))
)

users_silver = users_silver.withColumn(
    "address",
    trim(lower(col("address")))
)

users_silver = users_silver.withColumn(
    "yearly_income",
    col("yearly_income").cast("double")
)

users_silver = users_silver.withColumn(
    "per_capita_income",
    col("per_capita_income").cast("double")
)

users_silver = users_silver.withColumn(
    "total_debt",
    col("total_debt").cast("double")
)

In [8]:
# Transformar cards

cards_silver = cards_spark.withColumn(
    "card_brand",
    trim(lower(col("card_brand")))
)

cards_silver = cards_silver.withColumn(
    "card_type",
    trim(lower(col("card_type")))
)

cards_silver = cards_silver.withColumn(
    "has_chip",
    trim(lower(col("has_chip")))
)

cards_silver = cards_silver.withColumn(
    "card_on_dark_web",
    trim(lower(col("card_on_dark_web")))
)

cards_silver = cards_silver.withColumn(
    "credit_limit",
    col("credit_limit").cast("double")
)

In [9]:
# Transformar transactions

transactions_silver = transactions_spark.withColumn(
    "use_chip",
    trim(lower(col("use_chip")))
)

transactions_silver = transactions_silver.withColumn(
    "merchant_city",
    trim(lower(col("merchant_city")))
)

transactions_silver = transactions_silver.withColumn(
    "merchant_state",
    trim(lower(col("merchant_state")))
)

transactions_silver = transactions_silver.withColumn(
    "amount",
    col("amount").cast("double")
)

## Convertir fechas

In [10]:
# Convertir fecha de transactions

transactions_silver = transactions_silver.withColumn(
    "date",
    coalesce(
        try_to_timestamp(
            col("date"),
            lit("dd/MM/yyyy HH:mm")
        ),
        try_to_timestamp(
            col("date"),
            lit("MM/dd/yyyy HH:mm")
        )
    )
)

## Eliminar registros duplicados

In [11]:
# Eliminar duplicados

users_silver = users_silver.dropDuplicates(["id"])

cards_silver = cards_silver.dropDuplicates(["id"])

transactions_silver = transactions_silver.dropDuplicates(["id"])

## Imputar valores faltantes con Machine Learning

Se utiliza un modelo de vecinos cercanos para estimar los límites de crédito faltantes.

El modelo utiliza características de las tarjetas y la información financiera de los clientes.

Se compara el resultado con una imputación por mediana para seleccionar el método con menor error.

In [12]:
# Importar librerías

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    DoubleType
)

## Preparar los datos de entrenamiento

Se unen las características de las tarjetas con la información de los usuarios.

Los identificadores se utilizan para relacionar los registros, pero no forman parte de las variables predictoras.

In [13]:
# Preparar información de usuarios

users_features = users_silver.select(
    col("id").alias("client_id"),
    "current_age",
    "yearly_income",
    "total_debt",
    "credit_score"
)


# Unir información

cards_features = (
    cards_silver
    .select(
        "id",
        "client_id",
        "card_brand",
        "card_type",
        "has_chip",
        "num_cards_issued",
        "credit_limit"
    )
    .join(
        users_features,
        "client_id",
        "left"
    )
)


# Convertir datos a pandas

cards_pd = cards_features.toPandas()


# Definir variables

numeric_columns = [
    "num_cards_issued",
    "current_age",
    "yearly_income",
    "total_debt",
    "credit_score"
]

categorical_columns = [
    "card_brand",
    "card_type",
    "has_chip"
]

feature_columns = numeric_columns + categorical_columns


# Preparar valores faltantes

cards_pd[numeric_columns] = (
    cards_pd[numeric_columns]
    .astype("float64")
    .replace([np.inf, -np.inf], np.nan)
)

for column in categorical_columns:
    cards_pd[column] = cards_pd[column].map(
        lambda value: (
            np.nan
            if pd.isna(value) or str(value).strip() == ""
            else str(value).strip().lower()
        )
    )


# Separar registros

known_cards = cards_pd.loc[
    cards_pd["credit_limit"].notna()
    & cards_pd["client_id"].notna()
].copy()

missing_cards = cards_pd.loc[
    cards_pd["credit_limit"].isna()
    & cards_pd["id"].notna()
    & cards_pd["client_id"].notna()
].copy()

print("Tarjetas con límite conocido:", len(known_cards))
print("Tarjetas con límite faltante:", len(missing_cards))

Tarjetas con límite conocido: 6185
Tarjetas con límite faltante: 1


## Crear el modelo de vecinos cercanos

Se normalizan las variables numéricas y se codifican las variables categóricas.

Los clientes se separan entre entrenamiento y validación para evitar que las tarjetas de un mismo cliente aparezcan en ambos conjuntos.

In [14]:
# Separar variables

X = known_cards[feature_columns]
y = known_cards["credit_limit"]

groups = known_cards["client_id"]


# Dividir datos

split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    split.split(X, y, groups=groups)
)

X_train = X.iloc[train_index]
X_test = X.iloc[test_index]

y_train = y.iloc[train_index]
y_test = y.iloc[test_index]


# Preparar variables numéricas

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])


# Preparar variables categóricas

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])


# Crear transformaciones

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_columns
    ),
    (
        "categorical",
        categorical_pipeline,
        categorical_columns
    )
])


# Crear modelo

model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "knn",
        KNeighborsRegressor(
            n_neighbors=5,
            weights="distance"
        )
    )
])


# Entrenar modelo

model.fit(X_train, y_train)

print("Modelo entrenado")

Modelo entrenado


## Evaluar el modelo

Se calcula el error absoluto medio de las predicciones.

Se compara el modelo de vecinos cercanos con la mediana calculada sobre los datos de entrenamiento.

In [15]:
# Generar predicciones

knn_predictions = model.predict(X_test)

median_predictions = np.full(
    len(y_test),
    y_train.median()
)


# Calcular errores

knn_mae = mean_absolute_error(
    y_test,
    knn_predictions
)

median_mae = mean_absolute_error(
    y_test,
    median_predictions
)


# Comparar resultados

results = pd.DataFrame({
    "modelo": ["KNN", "Mediana"],
    "MAE": [knn_mae, median_mae]
})

display(results)


# Seleccionar método

selected_method = (
    "knn"
    if knn_mae < median_mae
    else "mediana"
)

print("Método seleccionado:", selected_method)

,modelo,MAE
0,KNN,5467.050082
1,Mediana,8219.635930


Método seleccionado: knn


## Completar los límites de crédito

Se aplica el método seleccionado a las tarjetas con límite de crédito faltante.

Se conserva una columna para identificar los valores estimados.

In [16]:
# Estimar valores faltantes

if not missing_cards.empty:

    if selected_method == "knn":

        model.fit(X, y)

        predictions = model.predict(
            missing_cards[feature_columns]
        )

    else:

        predictions = np.full(
            len(missing_cards),
            y.median()
        )


    # Preparar estimaciones

    estimated_cards = missing_cards[
        ["id", "client_id"]
    ].copy()

    estimated_cards["credit_limit_estimated"] = (
        np.round(predictions, 2)
    )

    display(estimated_cards)


    # Convertir estimaciones a Spark

    estimated_schema = StructType([
        StructField("id", LongType(), False),
        StructField(
            "credit_limit_estimated",
            DoubleType(),
            False
        )
    ])

    estimated_rows = [
        (
            int(row.id),
            float(row.credit_limit_estimated)
        )
        for row in estimated_cards.itertuples(index=False)
    ]

    estimated_spark = spark.createDataFrame(
        estimated_rows,
        schema=estimated_schema
    )


    # Actualizar cards

    cards_silver = (
        cards_silver
        .join(
            estimated_spark,
            "id",
            "left"
        )
        .withColumn(
            "credit_limit_imputed",
            col("credit_limit").isNull()
            & col("credit_limit_estimated").isNotNull()
        )
        .withColumn(
            "credit_limit_imputation_method",
            when(
                col("credit_limit_imputed"),
                lit(selected_method)
            ).otherwise(lit(None).cast("string"))
        )
        .withColumn(
            "credit_limit",
            coalesce(
                col("credit_limit"),
                col("credit_limit_estimated")
            )
        )
        .drop("credit_limit_estimated")
    )

    print("Límites de crédito completados")

else:

    print("No hay límites de crédito para imputar")

,id,client_id,credit_limit_estimated
6163,8453.0,928,11682.27


Límites de crédito completados


## Verificar valores faltantes

In [17]:
# Verificar valores faltantes

users_nulls = users_silver.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in users_silver.columns
])

cards_nulls = cards_silver.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in cards_silver.columns
])

transactions_nulls = transactions_silver.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in transactions_silver.columns
])

print("Valores faltantes en users:")
users_nulls.show()

print("Valores faltantes en cards:")
cards_nulls.show()

print("Valores faltantes en transactions:")
transactions_nulls.show()

Valores faltantes en users:
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
| id|current_age|retirement_age|birth_year|birth_month|gender|address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  0|          0|             0|         0|          0|     0|      0|       1|        0|                0|            0|         0|           0|               0|
+---+-----------+--------------+----------+-----------+------+-------+--------+---------+-----------------+-------------+----------+------------+----------------+

Valores faltantes en cards:
+---+---------+----------+---------+-----------+-------+---+--------+----------------+------------+--------------+--------------

## Guardar SILVER

In [18]:
# Guardar SILVER

users_silver.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/users.parquet"
)

cards_silver.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/cards.parquet"
)

transactions_silver.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/transactions.parquet"
)

print("Pipeline SILVER completado")

Pipeline SILVER completado


In [19]:
# Finalizar Spark

spark.stop()